# Complaint Management System — Implementation Walkthrough

This notebook documents, step by step, everything implemented in the backend so far.
Each step explains **what** was built, **why** it was designed that way, and ends with a
**runnable verification cell** so you can confirm the piece actually works against your
own `.env` credentials.

## The big picture

A RAG-based system for handling customer complaints:

```
                       ┌──────────────────────────────┐
                       │        FastAPI backend        │
  complaint ──────────▶│  (routing, retrieval, drafts) │
                       └───────┬──────────────┬───────┘
                               │              │
                   source of truth      rebuildable index
                               │              │
                       ┌───────▼──────┐ ┌─────▼────────┐
                       │   Supabase   │ │    Qdrant    │
                       │  (Postgres)  │ │ (hybrid vec) │
                       │ docs, chunks │ │ dense+sparse │
                       │ tickets, RLS │ │   points     │
                       └──────────────┘ └──────────────┘
```

**Core architectural decision:** Postgres (Supabase) is the *source of truth* for all text and
metadata. Qdrant is a *derived, rebuildable cache* of embeddings. If Qdrant is lost, it can be
rebuilt entirely from Postgres — never the other way round. This single decision drives the
write ordering, the idempotency design, and the failure-handling you'll see in Step 5.

## Steps covered

| Step | What was built | Where it lives |
|------|----------------|----------------|
| 1 | Project setup, config, TLS trust store | `pyproject.toml`, `app/config.py`, `app/__init__.py` |
| 2 | FastAPI app + health endpoints | `app/main.py`, `app/api/health.py` |
| 3 | Supabase schema + Row Level Security | `supabase/migrations/0001_init.sql`, `0002_rls.sql` |
| 4 | Qdrant collection (hybrid dense+sparse) | `scripts/create_qdrant_collection.py`, `app/services/vector_store.py` |
| 5 | Ingestion pipeline (chunk → Postgres → Qdrant) | `app/ingestion/chunkers.py`, `app/ingestion/pipeline.py` |
| 6 | Seed corpus ingestion | `scripts/seed.py`, `data/seed/` |

## Notebook setup

Two things every later cell depends on:

1. **Working directory** — `app/config.py` loads a *relative* `.env`, so we `chdir` to `backend/`.
2. **Import path** — the project is not pip-installed, so `backend/` is added to `sys.path`
   to make `app` importable (the same trick `scripts/*.py` use).

Importing anything from `app` also runs `app/__init__.py`, which injects the **OS trust store**
into Python's `ssl` module via `truststore`. This matters on machines with a TLS-intercepting
proxy/AV: its root CA lives in the Windows cert store but not in certifi's bundle, so without
this every HTTPS client (Supabase, Qdrant, OpenAI, LangSmith) fails with
`unable to get local issuer certificate`. It must run **before** any HTTPS client is constructed.

In [ ]:
import logging
import os
import sys
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(levelname)-8s %(name)s: %(message)s")
logger = logging.getLogger("notebook")

try:
    BACKEND_DIR = Path.cwd() if (Path.cwd() / "app").exists() else Path.cwd().parent
    assert (BACKEND_DIR / "app").exists(), f"Could not locate backend/ from {Path.cwd()}"
    os.chdir(BACKEND_DIR)                      # so Settings finds the relative .env
    sys.path.insert(0, str(BACKEND_DIR))       # so `app` is importable
    logger.info("Working directory: %s", BACKEND_DIR)
except Exception:
    logger.exception("Notebook setup failed — run this from backend/ or backend/notebooks/")
    raise

---
## Step 1 — Project setup & configuration

### Dependencies (`pyproject.toml`, managed with `uv`)

| Package | Role |
|---------|------|
| `fastapi` + `uvicorn` | HTTP API |
| `supabase` | Postgres + auth + storage client |
| `qdrant-client` | Vector database client |
| `langchain`, `langchain-openai`, `langchain-qdrant` | Embeddings + vector-store abstraction |
| `langchain-text-splitters` | Markdown-aware chunking |
| `fastembed` | Local BM25 sparse embeddings (no API cost) |
| `langgraph` | Agent graph (upcoming steps) |
| `langsmith` | Tracing/observability |
| `pydantic-settings` | Typed env configuration |
| `truststore` | OS trust store for TLS (see setup above) |

### Configuration — `app/config.py`

A single `Settings(BaseSettings)` class is the **single source of truth** for all environment
configuration. Key design points:

- **Required fields have no default** (`openai_api_key`, `qdrant_url`, `supabase_url`, ...) —
  a missing value fails **at startup** with a clear pydantic error, not as a confusing failure
  on the first request that needs it.
- **Tunables are env-overridable** (`dept_confidence_threshold=0.60`, `max_retrieval_attempts=2`)
  so eval-driven changes don't need a code release.
- `get_settings()` is wrapped in `@lru_cache` — the `.env` file is parsed once per process, and
  every module shares the same instance.
- Derived values are `@property`s (`supabase_jwks_url`, `cors_origins_list`) so the raw env
  stays simple strings.

In [ ]:
# Verify: load settings and show the non-secret fields.
try:
    from app.config import get_settings

    settings = get_settings()
    print(f"Main model:        {settings.openai_model_main}")
    print(f"Cheap model:       {settings.openai_model_cheap}")
    print(f"Embedding model:   {settings.embedding_model} ({settings.embedding_dims} dims)")
    print(f"Qdrant collection: {settings.qdrant_collection}")
    print(f"LangSmith project: {settings.langsmith_project}")
    print(f"Dept threshold:    {settings.dept_confidence_threshold}")
    print(f"CORS origins:      {settings.cors_origins_list}")
except Exception:
    logger.exception("Settings failed to load — check that backend/.env exists and is complete")
    raise

---
## Step 2 — FastAPI app & health endpoints

`app/main.py` uses an **app factory** (`create_app()`): CORS middleware is configured from
settings and routers are attached there, keeping the module importable without side effects.

`app/api/health.py` exposes two endpoints:

- **`GET /health`** — trivial liveness check, returns `{"status": "ok"}`.
- **`GET /health/deps`** — pings both external dependencies so a broken `.env` shows up
  *here*, not on the first real request:
  - **Supabase**: hits GoTrue's `/auth/v1/health` — chosen because it needs **no schema or
    tables to exist yet**, so it was safe to use before Step 3.
  - **Qdrant**: calls `get_collections()` — proves connectivity without requiring a
    collection to exist yet (that comes in Step 4).
  
  Each ping is wrapped in its own `try/except`, so one broken dependency reports
  `"error: ..."` while the other still reports `"ok"`, and the overall status becomes
  `"degraded"` instead of the endpoint crashing.

Run the server with:
```bash
cd backend
uv run uvicorn app.main:app --reload
```

In [ ]:
# Verify: exercise both endpoints in-process (no server needed) via FastAPI's TestClient.
try:
    from fastapi.testclient import TestClient

    from app.main import app

    with TestClient(app) as client:
        print("/health      →", client.get("/health").json())
        print("/health/deps →", client.get("/health/deps").json())
except Exception:
    logger.exception("Health check failed")
    raise

---
## Step 3 — Supabase schema & Row Level Security

Two versioned SQL migrations (both fully **re-runnable**: `IF NOT EXISTS`, `OR REPLACE`,
`ON CONFLICT`, `DROP POLICY IF EXISTS`):

### `0001_init.sql` — tables

| Group | Tables | Notes |
|-------|--------|-------|
| Identity | `profiles` | 1:1 with `auth.users`; auto-created by a `SECURITY DEFINER` trigger on signup so FKs never dangle |
| Taxonomy | `departments` | 12 fixed slugs seeded in the migration — the `description` column is fed verbatim into the (upcoming) department-classifier prompt |
| Knowledge base | `documents`, `chunks` | Source of truth that Qdrant mirrors; `chunks` has `UNIQUE (document_id, chunk_index)` — **the idempotency key** for re-ingest |
| Chat | `chat_sessions`, `messages` | `messages.user_id` is denormalised so RLS is a column compare, not a subquery per row |
| Ops | `ingestion_jobs` | One row per ingest run: status, counts, timings, error |
| Signal | `feedback`, `draft_feedback` | `UNIQUE (message_id, user_id)` — changing your mind is an upsert, not a second row |
| Tickets | `tickets`, `drafts`, `dept_responses`, `ticket_events` | Schema now, UI later; `ticket_events` is an append-only audit log |

Notable decisions:

- **No `vector` extension.** Vectors live in Qdrant; Postgres stores text + metadata only.
- `updated_at` is maintained by a shared **database trigger** (`set_updated_at()`), never
  trusted to writers.
- `documents.status` lifecycle: `pending → processing → indexed | failed` (plus `deleting`
  for crash-safe delete/replace later).
- `drafts` stores `retrieved_cases` + `prompt_version` so failures can later be attributed:
  *did retrieval pick the wrong evidence, or did drafting misuse it?*

### `0002_rls.sql` — Row Level Security

RLS is enabled on **every** table ("a single table without it is a hole in the floor").
The policies describe what a *logged-in agent's JWT* may do; the **service-role key bypasses
RLS entirely**, which is exactly why the ingestion pipeline needs no INSERT policy on `chunks`.

- Knowledge base (`documents`, `chunks`): **shared across the org** — everyone reads.
- Chat (`chat_sessions`, `messages`, `feedback`): **strictly per-user** (`user_id = auth.uid()`).
- `is_admin()` helper is `SECURITY DEFINER` so an admin check inside a policy on `profiles`
  doesn't recurse into itself.
- Role escalation is blocked: the `profiles` update policy's `WITH CHECK` pins `role` to its
  current value.
- `ticket_events` is append-only **by omission**: with RLS on, a command with no policy is
  denied — so simply not writing UPDATE/DELETE policies makes the audit log immutable.

### Backend client — `app/services/supabase_client.py`

`get_supabase()` builds one cached client using the **service-role (secret) key** — deliberate,
because ingestion writes org-wide rows and can't run under one user's policies. The flip side:
that key must **never** reach a browser; user-facing reads will use the publishable key + JWT.

In [ ]:
# Verify: read the departments taxonomy through the service-role client.
try:
    from app.services.supabase_client import get_supabase

    supabase = get_supabase()
    departments = supabase.table("departments").select("id,name,mailbox").order("id").execute()
    print(f"{len(departments.data)} departments:")
    for dept in departments.data:
        print(f"  {dept['id']:<15} {dept['name']:<20} {dept['mailbox']}")
except Exception:
    logger.exception("Supabase query failed — have the migrations been applied?")
    raise

---
## Step 4 — Qdrant collection (hybrid search)

### Why a script, not dashboard clicks

The vector index is derived, rebuildable state — but its *shape* is schema. So it's created by
a versioned, idempotent script (`scripts/create_qdrant_collection.py`), exactly like the SQL
migrations. Safe to re-run: an existing collection is left untouched.

### Collection `complaint_kb_v1`

| Vector | Config | Purpose |
|--------|--------|--------|
| `dense` (named) | 1536 dims, cosine | Semantic similarity via `text-embedding-3-small` |
| `sparse` (named) | **IDF modifier** | BM25 keyword matching via `fastembed` (local, free) |

The IDF modifier is essential: without it Qdrant scores raw term frequencies and BM25
retrieval **quietly degrades** — no error, just worse results.

### The `metadata.` payload gotcha

All writes/reads go through `langchain_qdrant.QdrantVectorStore`, which **hardcodes** its
payload shape — chunk text under `page_content`, all metadata nested under one `metadata` key:

```json
{
  "page_content": "<chunk text>",
  "metadata": {"doc_id": "...", "doc_type": "...", "department": "...", "category": "..."}
}
```

**Consequence:** every payload filter and index must use the dotted path
(`metadata.department`), never the flat name (`department`). Qdrant handles nested paths
natively so this costs nothing — but a filter written against the flat name **silently matches
zero points**. Keyword payload indexes are created on `metadata.doc_id`, `metadata.doc_type`,
`metadata.department`, `metadata.category`.

### `app/services/vector_store.py` — one definition, no drift

The vector names and indexed fields are constants in this module; both the creation script and
the ingestion pipeline import them, so the collection that gets *created* and the one that gets
*written to* can never drift apart. Cached factories:

- `get_qdrant_client()` — sync client (both callers are sync; `/health/deps` keeps its own async one)
- `get_dense_embeddings()` — `OpenAIEmbeddings` via LangChain, so LangSmith traces embed calls for free
- `get_sparse_embeddings()` — `FastEmbedSparse("Qdrant/bm25")`; first call downloads a ~50 MB ONNX model, cached per process
- `get_vector_store()` — `QdrantVectorStore` in `RetrievalMode.HYBRID`; collection validation is left **on**, so a mismatched collection fails loudly here instead of silently writing wrong vectors

Run once with:
```bash
cd backend
uv run python scripts/create_qdrant_collection.py
```

In [ ]:
# Verify: read the collection back and confirm shape matches the design.
try:
    from app.services.vector_store import (
        DENSE_VECTOR_NAME,
        INDEXED_PAYLOAD_FIELDS,
        SPARSE_VECTOR_NAME,
        get_qdrant_client,
    )

    client = get_qdrant_client()
    info = client.get_collection(settings.qdrant_collection)
    params = info.config.params

    print(f"Collection:      {settings.qdrant_collection}")
    print(f"Dense vectors:   {params.vectors}")
    print(f"Sparse vectors:  {params.sparse_vectors}")
    print(f"Payload indexes: {sorted(info.payload_schema or {})}")
    print(f"Points stored:   {info.points_count}")

    assert DENSE_VECTOR_NAME in params.vectors, "dense vector name mismatch"
    assert SPARSE_VECTOR_NAME in params.sparse_vectors, "sparse vector name mismatch"
    assert set(INDEXED_PAYLOAD_FIELDS) <= set(info.payload_schema or {}), "missing payload index"
    print("\nCollection shape matches app/services/vector_store.py ✓")
except Exception:
    logger.exception("Collection check failed — run scripts/create_qdrant_collection.py first")
    raise

---
## Step 5a — Chunking strategies (`app/ingestion/chunkers.py`)

Two document types, two deliberately different strategies:

### `case` — no split at all

A resolved complaint is **semantically atomic**: separating the resolution from the complaint
it resolves destroys the retrieval unit — neither half answers *"how was this handled?"* alone.
So one case = one chunk = one Qdrant point. `build_case_text()` flattens a case record into a
labelled block (`COMPLAINT:` / `DEPARTMENT GUIDANCE:` / `RESOLUTION:`) — the labels are kept in
the embedded text on purpose, so a retrieval hit reads as *"this is the complaint, this is what
we did about it"*.

### `policy` — header-aware split, then size split

Policies are long-form and cited by clause (*"per warranty §2.3"*), so precision matters:

1. `MarkdownHeaderTextSplitter` splits on `#`/`##`/`###` (with `strip_headers=True`).
2. `RecursiveCharacterTextSplitter.from_tiktoken_encoder` enforces **~800 tokens per chunk
   with 100 overlap** — overlap exists so a clause straddling a boundary is still fully
   present in at least one chunk.
3. Each chunk gets its **heading breadcrumb** prepended
   (`Product Warranty Policy > 2. Manufacturing Defects > 2.3 ...`) — a bare paragraph about
   "structural failures" is useless to cite; the breadcrumb tells the agent which clause
   they're quoting.

Supporting pieces:

- `parse_frontmatter()` — hand-rolled `---` block parser (three flat string fields; pulling in
  pyyaml for that would be borrowing an undeclared transitive dependency).
- `count_tokens()` — cl100k_base count for `chunks.token_count`. **Never raises**: the count is
  diagnostic metadata, so a tokenizer problem must not fail an otherwise good ingest.

In [ ]:
# Demo: chunk a real seed policy and show the breadcrumbs.
try:
    from app.ingestion.chunkers import chunk_document, count_tokens, parse_frontmatter

    policy_path = BACKEND_DIR / "data" / "seed" / "policies" / "warranty-policy.md"
    meta, body = parse_frontmatter(policy_path.read_text(encoding="utf-8"))
    print(f"Frontmatter: {meta}\n")

    chunks = chunk_document("policy", body)
    print(f"{len(chunks)} chunks:")
    for i, chunk in enumerate(chunks):
        breadcrumb = chunk.split("\n", 1)[0]
        print(f"  [{i}] {count_tokens(chunk):>4} tokens | {breadcrumb}")

    print("\n--- first chunk, first 300 chars ---")
    print(chunks[0][:300])
except Exception:
    logger.exception("Chunking demo failed")
    raise

---
## Step 5b — Ingestion pipeline (`app/ingestion/pipeline.py`)

`ingest_document(document_id, raw_text)` takes one document end to end. The **write order is a
consistency protocol**, not arbitrary:

```
chunk → write `chunks` rows (Postgres) → embed + upsert points (Qdrant) → mark indexed
```

**Postgres before Qdrant**, because Postgres is the source of truth and Qdrant is rebuildable.
A crash in between leaves the document visibly stuck at `status='processing'`, and a re-run
finishes the job. The reverse order could leave *ghost chunks* — Qdrant points that retrieval
can find but Postgres cannot explain. That is the dangerous failure direction.

### Three layers of idempotency (re-runs are free and safe)

1. **Content-hash short-circuit** — if `status='indexed'` and the SHA-256 of the text matches
   `documents.content_hash`, return `skipped` immediately: zero OpenAI calls.
2. **Chunk upsert on `(document_id, chunk_index)`** — rows update in place; rows beyond the new
   chunk count are deleted so a document that *shrank* doesn't leave orphaned tail rows.
3. **Deterministic point ids** — `uuid5(NAMESPACE, f"{doc_id}:{chunk_index}:{chunk_hash}")`,
   so re-ingesting unchanged content **overwrites** points instead of duplicating them.
   The namespace UUID is fixed forever — changing it would orphan every existing point.

### Stale-point cleanup

Because a point id embeds the chunk's **content hash**, edited text produces a *new* id — the
old point would linger and stay retrievable. So the pipeline snapshots the document's existing
point ids (paged `scroll` on `metadata.doc_id`), upserts the new points, then deletes
`previous − new`.

### Bookkeeping & observability

- Every run inserts an `ingestion_jobs` row (`running → done | failed`) with chunk/point counts
  and timings.
- On failure, `_mark_failed()` records the error on both `documents` and `ingestion_jobs` —
  wrapped in its own `try/except` so if *Postgres itself* broke, the bookkeeping failure
  doesn't mask the original exception.
- `@traceable(project_name="<project>-ingest")` sends ingest traces to a **separate LangSmith
  project**, so they don't bury the chat traces read day to day.
- Embedding goes through `vector_store.add_documents()`, which **batches** the OpenAI calls —
  one round-trip per document, not one per chunk.

In [ ]:
# Demo: the idempotency primitives, no network needed.
try:
    from app.ingestion.pipeline import build_point_id, compute_content_hash

    text = "COMPLAINT:\nMy blender arrived with a cracked jar.\n\nRESOLUTION:\nReplaced under warranty."
    chunk_hash = compute_content_hash(text)
    doc_id = "11111111-2222-3333-4444-555555555555"

    id_run_1 = build_point_id(doc_id, 0, chunk_hash)
    id_run_2 = build_point_id(doc_id, 0, chunk_hash)
    id_edited = build_point_id(doc_id, 0, compute_content_hash(text + " Customer thanked us."))

    print(f"content hash:            {chunk_hash[:16]}...")
    print(f"point id (run 1):        {id_run_1}")
    print(f"point id (run 2):        {id_run_2}   ← identical: re-ingest overwrites")
    print(f"point id (edited text):  {id_edited}   ← new id: old point becomes stale and is deleted")
    assert id_run_1 == id_run_2 and id_run_1 != id_edited
except Exception:
    logger.exception("Idempotency demo failed")
    raise

---
## Step 6 — Seeding the corpus (`scripts/seed.py`)

The seed corpus lives on disk under `data/seed/`:

- `cases.json` — 20 resolved complaint cases (Path A direct + Path B escalated with
  `dept_guidance`)
- `policies/*.md` — 5 policy documents with `---` frontmatter (`title`, `department`)

Design points:

- **Derived document ids**: `uuid5(SEED_NAMESPACE, "case:C-1001")`. The schema has no natural
  unique key for a seed document, and without a stable id every re-run would create a second
  copy of the whole corpus. Deriving the primary key from the seed identity makes the entire
  script an upsert.
- `source='seed'` and `storage_path=NULL` mark these rows as fixture data rather than user
  uploads (nothing is uploaded to Supabase Storage).
- **Per-document isolation**: each ingest is wrapped in `try/except`, so one malformed case
  costs one failure count, not the other 24 documents.
- `--one` flag ingests a single case — the cheap *walking-skeleton* gate before spending on
  the full corpus.

```bash
cd backend
uv run python scripts/seed.py --one   # walking-skeleton check: one case
uv run python scripts/seed.py         # full 25-document corpus
```

Exit code is non-zero if any document failed, so it composes with CI later.

In [ ]:
# Verify: what the seed actually produced in Postgres.
try:
    docs = (
        supabase.table("documents")
        .select("title,doc_type,department_id,status,chunk_count")
        .eq("source", "seed")
        .order("doc_type")
        .execute()
    )
    if not docs.data:
        print("No seed documents yet — run: uv run python scripts/seed.py")
    else:
        by_status: dict[str, int] = {}
        for d in docs.data:
            by_status[d["status"]] = by_status.get(d["status"], 0) + 1
        print(f"{len(docs.data)} seed documents — status breakdown: {by_status}\n")
        for d in docs.data:
            print(f"  [{d['doc_type']:<6}] {d['status']:<9} {d['chunk_count']:>2} chunks  {d['title']}")
except Exception:
    logger.exception("Seed inspection failed")
    raise

---
## End-to-end check — hybrid retrieval over the seeded corpus

The final proof that every layer works together: run a **hybrid (dense + sparse) similarity
search** through the same `QdrantVectorStore` the pipeline writes through, with a payload
filter on the **dotted** metadata path (the Step 4 gotcha in action).

> Costs one embeddings API call for the query (plus one on first `get_vector_store()` call
> for its collection validation).

In [ ]:
try:
    from qdrant_client import models

    from app.services.vector_store import get_vector_store

    store = get_vector_store()
    query = "customer wants a refund for a double charge on their invoice"

    # NOTE the dotted path — 'department' alone would silently match nothing.
    dept_filter = models.Filter(
        must=[models.FieldCondition(key="metadata.department", match=models.MatchValue(value="billing"))]
    )

    results = store.similarity_search_with_score(query, k=3, filter=dept_filter)
    if not results:
        print("No results — has the seed been run?")
    for doc, score in results:
        m = doc.metadata
        print(f"score={score:.4f}  [{m['doc_type']}] {m['title']}")
        print(f"   {doc.page_content[:150].replace(chr(10), ' ')}...\n")
except Exception:
    logger.exception("Retrieval check failed — needs a created collection and a seeded corpus")
    raise